# Reclaim Value: train the material recognition models

Run this in **Google Colab** with a GPU: *Runtime → Change runtime type → T4 GPU*. Then run the cells **from top to bottom**.

You will get three files:

| File | What it is |
|---|---|
| `materials.pt` | detector: finds pieces of steel, concrete, brick, wood, tiles and glass in a photo |
| `materials_thresholds.json` | the best confidence cut-off for each material, tuned on the validation split |
| `materials_cls.pt` | whole-photo classifier: a second opinion, and recognises "other" (not a building material) |

Copy them into `ai-service/weights/` on your laptop and restart the AI service. The app then uses them instead of demo mode.

Budget about **2-3 hours**: about 20 minutes for downloads, 1-2 hours of training, plus time to take your own photos.

In [ ]:
!nvidia-smi -L   # should print a Tesla T4. If it errors, change the runtime type to GPU.
from google.colab import drive
drive.mount('/content/drive')

## Step 1 - Upload the code
In the left sidebar open **Files** and upload `reclaim-value.zip` (the whole project) to `/content`. Then run:

In [ ]:
!cd /content && unzip -oq reclaim-value.zip && ls reclaim-value
%cd /content/reclaim-value/ai-service/training
!pip -q install -r requirements.txt

## Step 2 - Download the public datasets

**2a. Roboflow** (several small construction-material datasets). Make a free account at roboflow.com, then go to *Settings → API keys* and copy your key.
In Colab, open the **key icon** in the left sidebar, add a secret named `ROBOFLOW_API_KEY`, paste the key and switch on notebook access.

Open each dataset page first and check its latest **version number**; change the number after `:` if it differs.

In [ ]:
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
!python download_roboflow.py data/raw \
  iaac/constrcution-materials:2 \
  enzo-guo-jfodj/cd_obb:1 \
  robocount/steel-construction-materials:1 \
  josiahs-project/construction-waste-fx7td:1 \
  bricks10/timber_brick_log_rebar \
  waste-detection-gt6wy/clear-broken-glass-0fj4v \
  vinoth-pujdr/construction_materials

**Glass and steel are the weak spots.** The cell above already adds three sets for them (no version number = newest version):
- `bricks10/timber_brick_log_rebar`: rebar (steel), brick and timber boxes
- `waste-detection-gt6wy/clear-broken-glass-0fj4v`: broken glass boxes
- `vinoth-pujdr/construction_materials`: an **Indian** photo set (TMT bars, construction glass, bricks, timber, plywood). It is a classification set, so it is saved in `data/raw_cls/` and used for the classifier in step 6.

To add another set you find on Roboflow Universe, put it below as `workspace/project` and run this cell.

In [ ]:
EXTRA_SETS = ''   # e.g. 'someone/window-glass' - leave empty if none
if EXTRA_SETS:
    !python download_roboflow.py data/raw {EXTRA_SETS}

**2b. CDW-Seg**: real photos of cluttered skip bins (concrete, timber, steel), plus plastic and cardboard that become the "other" class.
If the automatic download fails, open DOI 10.6084/m9.figshare.28573229 in a browser, download the zip, upload it to `data/cdwseg_raw/` and run only the second cell.

In [ ]:
import requests, os
os.makedirs('data/cdwseg_raw', exist_ok=True)
info = requests.get('https://api.figshare.com/v2/articles/28573229', timeout=60).json()
print(info.get('title'), '| licence:', (info.get('license') or {}).get('name'))
for f in info['files']:
    print('downloading', f['name'], round(f['size'] / 1e6), 'MB')
    with requests.get(f['download_url'], stream=True, timeout=900) as r, open('data/cdwseg_raw/' + f['name'], 'wb') as out:
        for chunk in r.iter_content(1 << 20):
            out.write(chunk)

In [ ]:
!cd data/cdwseg_raw && for z in *.zip; do unzip -oq "$z"; done
!python prepare_cdwseg.py data/cdwseg_raw data/cdwseg_yolo data/negatives/cdwseg

**2c. CODD (optional, recommended):** 3,000+ real C&D photos with bricks, concrete, tiles and wood. Download it by hand from https://data.mendeley.com/datasets/wds85kt64j/3, upload and unzip it to `data/codd`, then run:

In [ ]:
import os
if os.path.isdir('data/codd'):
    !python prepare_codd.py data/codd data/codd_yolo
else:
    print('CODD not uploaded - skipping')

## Step 3 - Your own photos (optional)
Public data comes from Cyprus, Australia, the USA and one Indian set. **You can skip this step**; the model still trains.
Photos from your own sites would make it more accurate on Indian sites:
1. Take 100-200 phone photos per material, label boxes in Roboflow with `steel`, `concrete`, `brick`, `wood`, `tiles`, `glass`, export as **YOLOv8** and upload as `data/mine`.
2. Keep ~30 more per material, never labelled or trained on, in `real_test/<material>/` (plus `real_test/other/`) for an honest real-world accuracy.

## Step 4 - Merge everything into the six classes and split without leakage
`merge_yolo.py` maps every dataset's labels onto the six materials and drops ambiguous ones such as "brickconcrete".
`resplit_by_source.py` keeps all augmented copies of a photo in one split, so test scores are honest.
**Read the printed box counts:** any material with fewer than about 300 boxes will be weak.

In [ ]:
import glob, os
srcs = sorted(glob.glob('data/raw/*'))
for extra in ('data/codd_yolo', 'data/cdwseg_yolo', 'data/mine'):
    if os.path.isdir(extra):
        srcs.append(extra)
print('datasets:', srcs)
!python merge_yolo.py data/merged {' '.join(srcs)}
!python resplit_by_source.py data/merged data/unified

## Step 5 - Train the detector (about 1-2 hours)
If a material has **0** boxes (usually glass), training stops with an error. Either add data for it, or put `--allow-missing-classes` at the end of the command. The model then never predicts that material, and the app asks the user to pick it.
If you see *CUDA out of memory*, change `--batch 16` to `--batch 8`.

In [ ]:
!python train_yolo.py data/unified/data.yaml materials --model yolo11s.pt --epochs 60 --imgsz 640 --batch 16 --patience 15 --strong-aug

Check the printed `mAP50` and per-class scores. The test split comes from the same public datasets, so treat this as an optimistic upper bound. Step 7 gives the honest number.

## Step 6 - Train the whole-photo classifier (about 15-30 minutes)

In [ ]:
import os, glob
neg = '--negatives data/negatives' if os.path.isdir('data/negatives') else ''
!python make_cls_dataset.py data/unified data/cls {neg}
cls_sets = ' '.join(glob.glob('data/raw_cls/*'))
if cls_sets:
    !python add_cls_folders.py data/cls {cls_sets}
!python train_cls.py data/cls materials_cls --model yolo11s-cls.pt --imgsz 320 --epochs 40

## Step 7 - Measure accuracy through the app's code
This runs held-out photos through the same function the app uses.
- With your own `real_test/` photos: this is **real-world accuracy**. Quote it in the report.
- Without them, it uses the **held-out public test split** (never trained on, split by original photo). In the report, call it *accuracy on held-out public data*, not real-site accuracy.

In [ ]:
import os
if os.path.isdir('real_test'):
    print('=== Real-world accuracy (your own photos) ===')
    !python eval_real_photos.py real_test --csv errors.csv
else:
    print('=== Accuracy on HELD-OUT PUBLIC test photos (no own photos provided) ===')
    !python eval_real_photos.py data/cls/test --csv errors.csv

Quick check on any single photo (upload one to `/content/test.jpg`):

In [ ]:
import sys, json
sys.path.insert(0, '..')
import vision
state = vision.load_detector()
print(json.dumps(state.public(), indent=1))
if os.path.exists('/content/test.jpg'):
    r = vision.analyze_image(open('/content/test.jpg', 'rb').read())
    print({k: r[k] for k in ('material', 'confidence', 'needs_manual', 'reason', 'alternatives', 'model', 'sources')})

## Step 8 - Save the model files
They are copied to Google Drive (`MyDrive/reclaim_weights`) and also downloaded to your computer.

In [ ]:
!mkdir -p /content/drive/MyDrive/reclaim_weights
!cp ../weights/materials.pt ../weights/materials_thresholds.json ../weights/materials_cls.pt /content/drive/MyDrive/reclaim_weights/
!ls -la /content/drive/MyDrive/reclaim_weights
from google.colab import files
for f in ('materials.pt', 'materials_thresholds.json', 'materials_cls.pt'):
    files.download(f'../weights/{f}')

## Step 9 - Use it in the app (on your laptop)
1. Put the three files in `reclaim-value/ai-service/weights/`.
2. In the AI service's virtual environment, run `pip install ultralytics` once.
3. Restart the service: `uvicorn main:app --port 8000`.
4. Open http://localhost:8000/health. `material_model.mode` should say **trained**, and `detector.loaded` and `classifier.loaded` should both be **true**.
5. In the app, Capture now shows "Trained model: …% confident" instead of the demo-mode warning.